# Worksheet 7: Agent-Based Simulation - Decisions for the Decade

This worksheet covers a basic agent-based model. The scenario is based on the [Decisions for the Decade](https://gca.org/wp-content/uploads/2026/04/Decisions-for-the-Decade.pdf) game. Click on the link to read more about the game. Here is a summary:

1. A country is made up of regions, that are administered by governors.
2. The country is susceptible to environmental disasters: floods and droughts.
3. To protect against these disasters, at the beginning of each decade, each governor is given 10 beans. Beans can be used to invest in:
   * Flood protection (umbrellas)
   * Drought protection (buckets)
   * Investment (prosperity)
4. A disaster occurs in each *year* according to pre-defined probabilities.
5. Each time a disaster occurs, a corresponding bean is used up. If there are no protection beans available for that disaster, all prosperity is used up, and a crisis occurs.

The game is played over 3 decades. The following cells define the agents and model to begin with. Read the code to understand and confirm that you understand what is going on.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from mesa import Agent, Model, batch_run
#from mesa.time import RandomActivation
from mesa.datacollection import DataCollector

In [ ]:
class Governor(Agent):
    """A provincial governor making resource allocation decisions."""
    
    def __init__(self, model, strategy='balanced'):
        super().__init__(model)
        self.strategy = strategy  # 'balanced', 'aggressive', 'conservative'
        
        # Track state
        self.umbrella = 0           # flood protection beans
        self.bucket = 0             # drought protection beans
        self.prosperity = 0         # development beans
        self.crises = 0             # cumulative crises
        self.prosperity_points = 0  # points earned (only if no crisis that decade)

        self.allocate_beans()
        
    def allocate_beans(self):
        """Allocate beans based on strategy and climate information."""
       
        if self.strategy == 'balanced':
            # Allocate equally: 3 umbrella, 3 bucket, 4 prosperity
            self.umbrella = 3
            self.bucket = 3
            self.prosperity = 4
        elif self.strategy == 'aggressive':
            # Maximize prosperity, minimal protection: 1, 1, 8
            self.umbrella = 1
            self.bucket = 1
            self.prosperity = 8
        elif self.strategy == 'conservative':
            # Maximize protection: 5 umbrella, 5 bucket, 0 prosperity
            self.umbrella = 5
            self.bucket = 5
            self.prosperity = 0
        else:
            raise ValueError('Unknown strategy')

    def update_beans(self, event):
        if event == 0: 
            # flood, reduce umbrella, update crises
            if self.umbrella == 0:
                self.crises += 1
                self.prosperity = 0
                print("Crisis!")
            else:
                self.umbrella -= 1
        elif event == 1:
            # drought, reduce bucket, update crises
            if self.bucket == 0:
                self.crises += 1
                self.prosperity = 0
                print("Crisis!")
            else:
                self.bucket -= 1
        else:
            # nothing, do nothing
            pass

        if (int(self.model.time) % 10) == 0:
            self.prosperity_points += self.prosperity
            self.allocate_beans()

In [4]:
class DecadesModel(Model):
    """Simulation of the Decisions for the Decade game."""
    
    def __init__(self, n_governors=4, strategies=None, num_decades=3):
        super().__init__()
        self.n_governors = n_governors
        self.num_decades = num_decades
        self.current_decade = -1
        self.event_dict = {0: 'flood', 1: 'drought', 2: 'nothing'}
        
        # If strategies not provided, use balanced for all
        if strategies is None:
            strategies = ['balanced'] * n_governors

        Governor.create_agents(model=self, n=n_governors, strategy = strategies)
        
        # Climate information per decade
        self.climate_pmf = np.array([[1/6, 1/6, 2/3], 
                                     [3/8, 1/8, 1/2],
                                     [0.35, 0.25, 0.40]])

    def get_climate_pmf(self, decade):
        if decade < 0 or decade >= len(self.climate_pmf):
            raise IndexError(
                f"No climate info for decade {decade}: climate_pmf has "
                f"{len(self.climate_pmf)} rows (num_decades={self.num_decades})."
            )
        return self.climate_pmf[decade]

    def step(self):
        if ((int(self.time-1) % 10 == 0) and (self.time > 0)):

            if self.current_decade + 1 >= self.num_decades:
                print("All decades complete - stopping.")
                return
            print("entering new decade")
            self.current_decade += 1
        event_generated = self.rng.choice([0, 1, 2], size= (1,), p = self.get_climate_pmf(self.current_decade))
        self.agents.shuffle_do("update_beans", event=event_generated[0])

## Ex. 1: Run the model

Run the model for 30 steps. Add comments where you need to, to ensure everything is running correctly. 

## Ex. 2: Data collection

Add a data collector step, that reports the attributes from each agent, at every step of the simulation.

## Ex. 3: Batch run

Create a batch run to:

- Run 100 iterations. In each iteration, the model should contain 3 governors - one using each strategy (`'balanced'`, `'aggressive'`, `'conservative'`).
- Use a different rng for each run.

Use the output to estimate a 95% CI for:
- the mean number of crises that each type of governor experiences over 3 decades.
- the mean number of prosperity points that each strategy yields

**Hint:** `batch_run` passes `rng` straight through to `model_cls(...)` for each run. Check what `DecadesModel.__init__` currently accepts, and what it forwards to `super().__init__()` - you may need to change both before a per-run rng actually takes effect.

**Hint:** If `results` comes back as an empty list even though the model clearly ran (you still see the print statements), the run itself isn't the problem. Look at what your `DataCollector` from Ex. 2 is actually tracking, and check both kinds of reporters it supports.

## Ex. 4: Extending the simulation

This section is for you to think about the scenario, and consider it may be extended.

- List 3 ways in which you can make the model more realistic.
- List 2 additional questions you would like to answer using the models.

*There is no need to code in this section; just ponder!*